In [22]:
# Simple Example

from caepher import generate_key_pair, encrypt, decrypt, key_string
import random

random.seed(4210)

message = "This is a very secret message."
private_key, public_key = generate_key_pair(rule_depth=3)

print(f"We generated the key pair:")
print(f"  -> Private Key: {key_string(private_key)}")
print(f"  -> Public Key: {key_string(public_key)}")
print()

encrypted_message = encrypt(public_key, bytes(message, encoding="utf-8"))
decrypted_message = decrypt(private_key, encrypted_message)
decrypted_string = decrypted_message.decode("utf-8")

print(f"And we received the following message: {decrypted_string}")

We generated the key pair:
  -> Private Key: Q0FPLTAuMTpwcnYAAAAAAAAFAA8AAwADAAAAAAAAAACi/10tjdKNWVmVWQ==
  -> Public Key: Q0FPLTAuMTpwdWIAAAAAAAAPAAMAAAAAAAAAAAAAABTPDwAdAP//AAD/JxH8D/8Uz/8PHf8MHwAAJwAAH///AAAAJ///APID8v8n7gPwAOsw//AAANL////YABH8/w/rMPD/4v8AABT8D////wDS6zAA8B3/DB8AACcAAB///xTPDwD//y0AFPwP/wAA/y0Uz/8PDC8t/wAAJwD//wAvFM8PAB0A//8AAP8nEfwP/wAAJwAA8f//FPz/Dx3/DPEAAAAn//8A8gPy/yfuA/AA6zD/8AAA0v///9gAEfz/DxTPDwAdAP//FPwP/wAA/y0AACcAAPH//xT8/w8d/wzxAAAAJ///APLrA/AADPL/Lesw//AAANL/FPz/D///0gAUzw8AHQD//wAA/ycR/A//FM//Dx3/DB8AACcAAB///+sw8P8AANL///8A2BH8D/8AACcA//8A8gPyJ//uAwDw6zDw/+L/AAAU/A////8A0uswAPAd/wwfAAAnAAAf//8Uzw8A//8tABT8D/8AAP8tFM//DwwvLf8AACcA//8ALxTPDwAdAP//AAD/JxH8D/8AACcAAPH//xT8/w8d/wzx6zDw/wAA0v///wDYEfwP/wAAJwD//wDyA/In/+4DAPDrMPD/4v8AABT8D////wDSAAAnAADx///rAwDwHf8M8RTPDwD//y0AFPwP/wAA/y0AACcA//8A8hT8/w8M8i3/FM8PAB0A//8AAP8nEfwP/xTP/w8d/wwfAAAnAAAf//8AAAAn//8A8gPy/yfuA/AA6zD/8AAA0v///9gAEfz/DxTPDwAdAP//FPwP/wAA/y0Uz/8PHf8MHwAAJwAAH///6zDw/wAA0v8U/A////8A0uswA

In [23]:
# This example shows more of the inner workings

import random
from caepher import get_random_reversible_rules, int_to_bytes, make_public_key, make_inverse_map, apply_rules, apply_once

N = 5  # neighborhood size
M = 40  # number of rules
L = 15 # CA length
R = 1  # repetitions

# number of possible CA states = 2^L
space = 1 << L

# get random reversible rules with L=15 from data/reversible.csv
rules = get_random_reversible_rules(L, n=M)
print(f"M={M} Selected Rules:\t\t\t\t{rules}")

# this is not exactly the private key generated above, but essentially our private key is the list of rules
rule_bytes = b''.join([
    int_to_bytes(rule, 1 << N)
    for rule in rules
])
print(f"Private Key:\t\t\t\t\t{rule_bytes.hex()}")

# again not exactly the public key generated above; our public key is a set of CA rules on 2^L states,
# which enacts the full composite action of all M rules above on an L-bit CA
composed_ca_rule = make_public_key(rules, L, N)
print(f"Public key size =\t\t\t\t{space} bits")

# uncomment this line to see public key
# print(f"Public key =\n{int_to_bytes(composed_ca_rule, space)}")

# random integer message
message = random.randrange(space)
print(f"Plaintext:\t\t\t\t\t\t{message:0{L}b}")

# encode by repeatedly applying each N=5 rule
encrypted = message
for _ in range(R):
    encrypted = apply_rules(encrypted, rules, L, N)
print(f"Encoded by all M N=5 rules:\t\t{encrypted:0{L}b}")

encrypted_compare = message
for _ in range(R):
    encrypted_compare = apply_once(encrypted_compare, composed_ca_rule, L)
print(f"Encoded by a single L=15 rule:\t{encrypted_compare:0{L}b}")
print(f"\t\t\t\t\t→ Match?\t{encrypted == encrypted_compare}")

## we can construct our inverse map from our list of rules
inverse = make_inverse_map(rules, L, N)
decrypted = encrypted
for _ in range(R):
    decrypted = inverse[decrypted]
print(f"Decoded:\t\t\t\t\t\t{decrypted:0{L}b}")
print(f"\t\t\t\t→ Recovered?\t{decrypted == message}")



M=40 Selected Rules:				[1499026841, 1431656793, 4042268400, 865678131, 2982392244, 3743481600, 3429289164, 252656655, 4278253320, 3743481600, 2795940518, 553454835, 3435960780, 13696815, 16318215, 3429682284, 1716872549, 2969521920, 3429666156, 268496895, 2594876074, 865285011, 3435973740, 4278254340, 252702735, 2594810538, 921712323, 3538955760, 1258337535, 4244832000, 1701205333, 1722115689, 3023877300, 234942975, 267390795, 4042276080, 3435961548, 252702735, 3435961548, 1431656793]
Private Key:					5959559955555959f0f01ef033993333b1c3b1b4df20ff00cc66cccc0f0f3c0fff00f708df20ff00a6a6aaa620fd0cf3cccc99cc00d0ff2f00f8ff07cc6ccc6c66556565b0ff4f00cc6c8d6c1000efff9aaaaaaa33933393cccccc6cff00fb040f0ff00f9aa9aaaa36f036c3d2f02df04b00b4fffd02ff006566555566a56669b43cb4b40e00f1ff0ff00f4bf0f03cf0cccc9ccc0f0ff00fcccc9ccc55555959
Public key size =				32768 bits
Plaintext:						101010110010001
Encoded by all M N=5 rules:		100110110010011
Encoded by a single L=15 rule:	100110110010011
					→ Match?	T

In [26]:
rules = get_random_reversible_rule(L)
print(rules)

[1449481557, 2594810538, 3538948080, 868432579]
